# Graded Lab: Agentic Workflows

In this lab, you will build an agentic system that generates a short research report through planning, external tool usage, and feedback integration. Your workflow will involve:

### Agents

* **Planning Agent / Writer**: Creates an outline and coordinates tasks.
* **Research Agent**: Gathers external information using tools like Arxiv, Tavily, and Wikipedia.
* **Editor Agent**: Reflects on the report and provides suggestions for improvement.

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to write your solution code or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the <span style="background-color: red; color: white; padding: 3px 5px; font-size: 16px; border-radius: 5px;">Submit assignment</span> button on the top right of the page.
---


### Research Tools

By importing `research_tools`, you gain access to several search utilities:

- `research_tools.arxiv_search_tool(query)` → search academic papers from **arXiv**  

  *Example:* `research_tools.arxiv_search_tool("neural networks for climate modeling")`

- `research_tools.tavily_search_tool(query)` → perform web searches with the **Tavily API**  

  *Example:* `research_tools.tavily_search_tool("latest trends in sunglasses fashion")`

- `research_tools.wikipedia_search_tool(query)` → retrieve summaries from **Wikipedia**  

  *Example:* `research_tools.wikipedia_search_tool("Ensemble Kalman Filter")`

Run the cell below to make them available.

In [3]:
# =========================
# Imports
# =========================

# --- Standard library 
from datetime import datetime
import re
import json
import ast


# --- Third-party ---
from IPython.display import Markdown, display
from aisuite import Client

# --- Local / project ---
import research_tools

In [4]:
import unittests

### Initialize client

Create a shared client instance for upcoming calls.

In [5]:
CLIENT = Client()

## Exercise 1: planner_agent

### Objective
Correctly set up a call to a language model (LLM) to generate a research plan.

### Instructions

1. **Focus Areas**:
   - Ensure `CLIENT.chat.completions.create` is correctly configured.
   - Pass the `model` and `messages` parameters correctly:
     - **Model**: Use `"openai:o4-mini"` by default.
     - **Messages**: Set with `{"role": "user", "content": user_prompt}`.
     - **Temperature**: Fixed at 1 for creative outputs.

### Notes

- The prompt is pre-defined and guides the LLM on task requirements.
- Only return a formatted list of steps — no extra text.

Focus on the LLM call setup to complete the task.

In [6]:
# GRADED FUNCTION: planner_agent

def planner_agent(topic: str, model: str = "openai:o4-mini") -> list[str]:
    """
    Generates a plan as a Python list of steps (strings) for a research workflow.

    Args:
        topic (str): Research topic to investigate.
        model (str): Language model to use.

    Returns:
        List[str]: A list of executable step strings.
    """

    
    # Build the user prompt
    user_prompt = f"""
    You are a planning agent responsible for organizing a research workflow with multiple intelligent agents.

    🧠 Available agents:
    - A research agent who can search the web, Wikipedia, and arXiv.
    - A writer agent who can draft research summaries.
    - An editor agent who can reflect and revise the drafts.

    🎯 Your job is to write a clear, step-by-step research plan **as a valid Python list**, where each step is a string.
    Each step should be atomic, executable, and must rely only on the capabilities of the above agents.

    🚫 DO NOT include irrelevant tasks like "create CSV", "set up a repo", "install packages", etc.
    ✅ DO include real research-related tasks (e.g., search, summarize, draft, revise).
    ✅ DO assume tool use is available.
    ✅ DO NOT include explanation text — return ONLY the Python list.
    ✅ The final step should be to generate a Markdown document containing the complete research report.

    Topic: "{topic}"
    """

    # Add the user prompt to the messages list
    messages = [{"role": "user", "content": user_prompt}]

    ### START CODE HERE ###

    # Call the LLM
    response = CLIENT.chat.completions.create( 
        # Pass in the model
        model=model,
        # Define the messages. Remember this is meant to be a user prompt!
        messages=messages,
        # Keep responses creative
        temperature=1, 
    )

    ### END CODE HERE ###

    # Extract message from response
    steps_str = response.choices[0].message.content.strip()

    # Parse steps
    steps = ast.literal_eval(steps_str)

    return steps

In [7]:
planner_agent("black holes")

["Use research agent to search Wikipedia for 'black holes' and extract fundamental definitions and historical context",
 'Use research agent to search arXiv for recent review papers on black hole astrophysics and thermodynamics',
 'Use research agent to collect key experimental and observational evidence on black holes from LIGO and the Event Horizon Telescope',
 'Use writer agent to draft an introduction summarizing definitions, history, and significance of black holes',
 'Use writer agent to draft a literature review section summarizing recent findings from arXiv papers',
 'Use writer agent to draft a section detailing observational evidence for black holes',
 'Use writer agent to draft a section explaining theoretical frameworks such as black hole thermodynamics and the information paradox',
 'Use editor agent to review and provide feedback on the introduction draft',
 'Use editor agent to review and provide feedback on the literature review draft',
 'Use editor agent to review and 

In [8]:
# Test your code!
unittests.test_planner_agent(planner_agent)

 All tests passed!


## Exercise 2: research_agent

### Objective
Set up a call to a language model (LLM) to perform a research task using various tools.

### Instructions

**Focus Areas**:

- **Creating a Custom Prompt**:
  - **Define the Role**: Clearly specify the role, such as "research assistant."
  - **List Available Tools** (as strings inside the prompt, not the actual functions):
    - Use `arxiv_tool` to find academic papers.
    - Use `tavily_tool` for general web searches.
    - Use `wikipedia_tool` for accessing encyclopedic knowledge.
  - **Specify the Task**: Include a placeholder in your prompt for defining the specific task that needs to be accomplished.
  - **Include Date Information**: Add a placeholder for the current date or time to provide context.

- **Creating Messages Dict**:
  - Ensure the `messages` are correctly set with `{"role": "user", "content": prompt}`.

- **Creating Tools List**:
  - Create a list of tools for use, such as `research_tools.arxiv_search_tool`, `research_tools.tavily_search_tool`, and `research_tools.wikipedia_search_tool`.

- **Correctly Setting the Call to the LLM**:
  - Pass the `model`, `messages`, and `tools` parameters accurately.
  - Set `tool_choice` to `"auto"` for automatic tool selection.
  - Limit interactions with `max_turns=6`.

### Notes

- The function provides pre-coded blocks where you need to replace placeholder values.
- The approach allows the LLM to use tools dynamically based on the task.

Focus on accurately setting the messages, tools, and LLM call parameters to complete the task.

In [9]:
type(research_tools)

module

In [10]:
help(research_tools)

Help on module research_tools:

NAME
    research_tools - # --- Standard library ---

FUNCTIONS
    arxiv_search_tool(query: str, max_results: int = 5) -> list[dict]
        Searches arXiv for research papers matching the given query.
    
    tavily_search_tool(query: str, max_results: int = 5, include_images: bool = False) -> list[dict]
        Perform a search using the Tavily API.
        
        Args:
            query (str): The search query.
            max_results (int): Number of results to return (default 5).
            include_images (bool): Whether to include image results.
        
        Returns:
            list[dict]: A list of dictionaries with keys like 'title', 'content', and 'url'.
    
    wikipedia_search_tool(query: str, sentences: int = 5) -> list[dict]
        Searches Wikipedia for a summary of the given query.
        
        Args:
            query (str): Search query for Wikipedia.
            sentences (int): Number of sentences to include in the summa

In [11]:
# GRADED FUNCTION: research_agent

def research_agent(task: str, model: str = "openai:gpt-4o", return_messages: bool = False):
    """
    Executes a research task using tools via aisuite (no manual loop).
    Returns either the assistant text, or (text, messages) if return_messages=True.
    """
    print("==================================")  
    print("🔍 Research Agent")                 
    print("==================================")

    current_time = datetime.now().strftime('%Y-%m-%d')
    
    ### START CODE HERE ###

    # Create a customizable prompt by defining the role (e.g., "research assistant"),
    # listing tools (arxiv_tool, tavily_tool, wikipedia_tool) for various searches,
    # specifying the task with a placeholder, and including a current_time placeholder.
    prompt = f"""
    You are a research assistant. 
    Given the task below, your job is to gather external information using tools like Arxiv, Tavily, and Wikipedia.
    
    Tools available:
    - Use arxiv_tool to find academic papers.
    - Use tavily_tool for general web searches.
    - Use wikipedia_tool for accessing encyclopedic knowledge.
    
    Task:
    {task}
    
    If needed, today date is {datetime.now().strftime("%Y-%m-%d")}.
    
    Once your analysis is complete, summarize:
    - The research from your analysis
    - The list of references you used
    """
    
    # Create the messages dict to pass to the LLM. Remember this is a user prompt!
    messages = [{"role": "user", "content": prompt}]

    # Save all of your available tools in the tools list. These can be found in the research_tools module.
    # You can identify each tool in your list like this: 
    # research_tools.<name_of_tool>, where <name_of_tool> is replaced with the function name of the tool.
    tools = [research_tools.arxiv_search_tool, 
             research_tools.tavily_search_tool,
             research_tools.wikipedia_search_tool]
    
    # Call the model with tools enabled
    response = CLIENT.chat.completions.create(  
        # Set the model
        model=model,
        # Pass in the messages. You already defined this!
        messages=messages,
        # Pass in the tools list. You already defined this!
        tools=tools,
        # Set the LLM to automatically choose the tools
        tool_choice="auto",
        # Set the max turns to 6
        max_turns=6
    )  
    
    ### END CODE HERE ###

    content = response.choices[0].message.content
    print("✅ Output:\n", content)

    
    return (content, messages) if return_messages else content  

In [12]:
research_agent("Research about Black holes", return_messages=True)

🔍 Research Agent
✅ Output:
 ### Research Summary on Black Holes

**Key Concepts:**

1. **Definition and Nature:**
   - Black holes are celestial objects with gravitational forces so strong that nothing, not even light, can escape from them. This boundary is known as the event horizon. Black holes result from the collapse of massive stars and are predicted by Albert Einstein's theory of general relativity.

2. **Types of Black Holes:**
   - *Stellar Black Holes*: Form through the gravitational collapse of massive stars. Their mass typically ranges from 5 to 100 times that of the Sun.
   - *Supermassive Black Holes*: Found at the centers of most galaxies, including the Milky Way, with masses ranging from hundreds of thousands to billions of solar masses.
   - *Primordial Black Holes*: Hypothetical black holes thought to have formed soon after the Big Bang.

3. **Formation and Growth:**
   - Black holes can form from supernova explosions, the merger of neutron stars, or other mechanisms s

('### Research Summary on Black Holes\n\n**Key Concepts:**\n\n1. **Definition and Nature:**\n   - Black holes are celestial objects with gravitational forces so strong that nothing, not even light, can escape from them. This boundary is known as the event horizon. Black holes result from the collapse of massive stars and are predicted by Albert Einstein\'s theory of general relativity.\n\n2. **Types of Black Holes:**\n   - *Stellar Black Holes*: Form through the gravitational collapse of massive stars. Their mass typically ranges from 5 to 100 times that of the Sun.\n   - *Supermassive Black Holes*: Found at the centers of most galaxies, including the Milky Way, with masses ranging from hundreds of thousands to billions of solar masses.\n   - *Primordial Black Holes*: Hypothetical black holes thought to have formed soon after the Big Bang.\n\n3. **Formation and Growth:**\n   - Black holes can form from supernova explosions, the merger of neutron stars, or other mechanisms such as primo

In [13]:
# Test your code!
unittests.test_research_agent(research_agent)

🔍 Research Agent
✅ Output:
 To provide a comprehensive response to your task, I will utilize three different tools to gather information: arXiv for academic papers, Tavily for general web searches, and Wikipedia for encyclopedic knowledge. I will search for key references on a relevant topic and summarize them. Kindly specify the topic of interest so that I can proceed effectively.
🔍 Research Agent
✅ Output:
 Based on the analysis of the gathered information, two seminal research papers were explored to gain insights into their contributions to the academic field. The first paper, titled "Back to the Seminal Deutsch Algorithm" by Giuseppe Castagnoli, revisits Deutsch's quantum algorithm, a fundamental work in quantum computing that emphasizes its interdisciplinary potential. This highlights the algorithm's critical role in advancing quantum mechanics and computational theory. The second work, "Discovering Seminal Works with Marker Papers" by Robin Haunschild and Werner Marx, discusses 

## Exercise 3: writer_agent

### Objective
Set up a call to a language model (LLM) for executing writing tasks like drafting, expanding, or summarizing text.

### Instructions

1. **Focus Areas**:
   - **System Prompt**:
     - Define `system_prompt` to assign the LLM the role of a writing agent focused on generating academic or technical content.
   - **System and User Messages**:
     - Create `system_msg` using `{"role": "system", "content": system_prompt}`.
     - Create `user_msg` using `{"role": "user", "content": task}`.
   - **Messages List**:
     - Combine `system_msg` and `user_msg` into a `messages` list.

### Notes

- The function is designed to produce well-structured text by setting the correct prompts.
- Temperature is set to 1.0 to allow for creative variance in the writing outputs.

Ensure the system prompt and messages are defined properly to achieve a structured output from the LLM.

In [14]:
# GRADED FUNCTION: writer_agent
def writer_agent(task: str, model: str = "openai:gpt-4o") -> str: # @REPLACE def writer_agent(task: str, model: str = None) -> str:
    """
    Executes writing tasks, such as drafting, expanding, or summarizing text.
    """
    print("==================================")
    print("✍️ Writer Agent")
    print("==================================")

    ### START CODE HERE ###
    
    # Create the system prompt.
    # This should assign the LLM the role of a writing agent specialized in generating well-structured academic or technical content
    system_prompt = f"""
    You are a writing agent specialized in generating well-structured academic or technical content.
    """

    # Define the system msg by using the system_prompt and assigning the role of system
    system_msg = {"role": "system", "content": system_prompt}

    # Define the user msg. In this case the user prompt should be the task passed to the function
    user_msg = {"role": "user", "content": task}

    # Add both system and user messages to the messages list
    messages = [system_msg, user_msg]
    
    ### END CODE HERE ###

    response = CLIENT.chat.completions.create(
        model=model, 
        messages=messages,
        temperature=1.0
    )

    return response.choices[0].message.content

In [15]:
writer_agent("black holes")

✍️ Writer Agent


'Black holes are among the most fascinating and enigmatic entities in the cosmos, characterized by their powerful gravitational forces from which nothing—not even light—can escape. Here, we provide an academic overview of their properties, formation, and significance in astrophysics:\n\n### 1. Formation of Black Holes\n\nBlack holes are formed from the remnants of massive stars that have undergone a gravitational collapse. When a star exhausts its nuclear fuel, it can no longer support itself against the inward pull of its own gravity. If the remaining mass is sufficient (generally greater than about three solar masses), the core collapses under its gravity, compressing its mass into an infinitely dense point known as a singularity, surrounded by an event horizon.\n\n- **Stellar Black Holes:** These form from the remnants of massive stars after supernova explosions.\n- **Supermassive Black Holes:** Found at the centers of galaxies, these are millions to billions of times more massive t

In [16]:
# Test your code!
unittests.test_writer_agent(writer_agent)

✍️ Writer Agent
 All tests passed!


## Exercise 4: editor_agent

### Objective
Configure a call to a language model (LLM) to perform editorial tasks such as reflecting, critiquing, or revising drafts.

### Instructions

1. **Focus Areas**:
   - **System Prompt**:
     - Define `system_prompt` to assign the LLM the role of an editor agent whose task is to reflect on, critique, or improve drafts.
   - **System and User Messages**:
     - Create `system_msg` using `{"role": "system", "content": system_prompt}`.
     - Create `user_msg` using `{"role": "user", "content": task}`.
   - **Messages List**:
     - Combine `system_msg` and `user_msg` into a `messages` list.

### Notes

- The editor agent is tailored for enhancing the quality of text by setting an appropriate role and task in the prompts.
- Temperature is set to 0.7, balancing creativity and coherence in editorial outputs.

Ensure the system prompt and messages are accurately set up to perform effective editorial tasks with the LLM.

In [17]:
# GRADED FUNCTION: editor_agent
def editor_agent(task: str, model: str = "openai:gpt-4o") -> str:
    """
    Executes editorial tasks such as reflection, critique, or revision.
    """
    print("==================================")
    print("🧠 Editor Agent")
    print("==================================")
    
    ### START CODE HERE ###

    # Create the system prompt.
    # This should assign the LLM the role of an editor agent specialized in reflecting on, critiquing, or improving existing drafts.
    system_prompt = f"""
    You are an editor agent specialized in reflecting on, critiquing, or improving existing drafts.
    """
    
    # Define the system msg by using the system_prompt and assigning the role of system
    system_msg = {"role": "system", "content": system_prompt}
    
    # Define the user msg. In this case the user prompt should be the task passed to the function
    user_msg = {"role": "user", "content": task}
    
    # Add both system and user messages to the messages list
    messages = [system_msg, user_msg]
    
    ### END CODE HERE ###
    
    response = CLIENT.chat.completions.create(
        model=model, 
        messages=messages,
        temperature=0.7 
    )
    
    return response.choices[0].message.content

In [18]:
# Test your code!
unittests.test_editor_agent(editor_agent)

🧠 Editor Agent
 All tests passed!


### 🎯 The Executor Agent

The `executor_agent` manages the workflow by executing each step of a given plan. It:

1. Decides **which agent** (`research_agent`, `writer_agent`, or `editor_agent`) should handle the step.
2. Builds context from the outputs of previous steps.
3. Sends the enriched task to the selected agent.
4. Collects and stores the results in a shared history.

👉 **Do not implement or modify this function.** It is already provided as the orchestration component of the multi-agent pipeline.

Notice that `planner_agent` might return a long list of steps. Because of this, the maximum number of steps is set to a maximum of 4 to keep running time reasonable.

In [19]:
agent_registry = {
    "research_agent": research_agent,
    "editor_agent": editor_agent,
    "writer_agent": writer_agent,
}

def clean_json_block(raw: str) -> str:
    """
    Clean the contents of a JSON block that may come wrapped with Markdown backticks.
    """
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
    return raw.strip()

In [1]:
def executor_agent(topic, model: str = "openai:gpt-4o", limit_steps: bool = True):

    plan_steps = planner_agent(topic)
    max_steps = 4

    if limit_steps:
        plan_steps = plan_steps[:min(len(plan_steps), max_steps)]
    
    history = []

    print("==================================")
    print("🎯 Editor Agent")
    print("==================================")

    for i, step in enumerate(plan_steps):

        agent_decision_prompt = f"""
        You are an execution manager for a multi-agent research team.

        Given the following instruction, identify which agent should perform it and extract the clean task.

        Return only a valid JSON object with two keys:
        - "agent": one of ["research_agent", "editor_agent", "writer_agent"]
        - "task": a string with the instruction that the agent should follow

        Only respond with a valid JSON object. Do not include explanations or markdown formatting.

        Instruction: "{step}"
        """
        response = CLIENT.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": agent_decision_prompt}],
            temperature=0,
        )

        raw_content = response.choices[0].message.content
        cleaned_json = clean_json_block(raw_content)
        agent_info = json.loads(cleaned_json)

        agent_name = agent_info["agent"]
        task = agent_info["task"]

        context = "\n".join([
            f"Step {j+1} executed by {a}:\n{r}" 
            for j, (s, a, r) in enumerate(history)
        ])
        enriched_task = f"""
        You are {agent_name}.

        Here is the context of what has been done so far:
        {context}

        Your next task is:
        {task}
        """

        print(f"\n🛠️ Executing with agent: `{agent_name}` on task: {task}")

        if agent_name in agent_registry:
            output = agent_registry[agent_name](enriched_task)
            history.append((step, agent_name, output))
        else:
            output = f"⚠️ Unknown agent: {agent_name}"
            history.append((step, agent_name, output))

        print(f"✅ Output:\n{output}")

    return history

In [20]:
# If you want to see the full workflow without limiting the number of steps. Set limit_steps to False
# Keep in mind this could take more than 10 minutes to complete
executor_history = executor_agent("The ensemble Kalman filter for time series forecasting", limit_steps=True)

md = executor_history[-1][-1].strip("`")  
display(Markdown(md))

🎯 Editor Agent

🛠️ Executing with agent: `research_agent` on task: search arXiv for recent papers on 'ensemble Kalman filter' in time series forecasting
🔍 Research Agent
✅ Output:
 ### Research Analysis

The search on arXiv for recent papers involving the "ensemble Kalman filter" (EnKF) in the context of time series forecasting yielded several relevant results. Below is a summary of the findings from these academic papers:

1. **Ensemble Kalman Filtering Meets Gaussian Process SSM for Non-Mean-Field and Online Inference**:
   - This paper integrates the ensemble Kalman filter with Gaussian process state-space models to improve inference accuracy and efficiency, especially in online settings. The approach addresses challenges in variational inference for nonlinear dynamical systems and manages to streamline parameterization, enhancing the model's applicability to real-time environments.

2. **LLM-Mixer: Multiscale Mixing in LLMs for Time Series Forecasting**:
   - Although not centered 


🛠️ Executing with agent: `research_agent` on task: compile a list of foundational and recent papers with titles, authors, and abstracts
🔍 Research Agent
✅ Output:
 ### Research Analysis

1. **Ensemble Kalman Filtering Meets Gaussian Process SSM for Non-Mean-Field and Online Inference**:
   - This paper focuses on integrating Gaussian process state-space models with ensemble Kalman filters (EnKF) to enhance performance under non-mean-field assumptions. It streamlines parameterization and supports online learning applications by using a closed-form approximation for the evidence lower bound (ELBO).

2. **The Geometric Unscented Kalman Filter**:
   - Introduces a new nonlinear estimation scheme named the geometric unscented Kalman filter (GUF). The GUF employs geometric unscented sampling to achieve high accuracy with good stability, avoiding issues like negative weights common in other filters.

3. **Accelerating the spin-up of Ensemble Kalman Filtering**:
   - Proposes a scheme to impr

### Research Summary

The research papers gathered focus predominantly on extensions and variations of the Kalman Filter, particularly the Ensemble Kalman Filter (EnKF), which has been modified or used in conjunction with other techniques for improved performance in various applications. Below are the insights from selected papers:

1. **Ensemble Kalman Filtering Meets Gaussian Process SSM for Non-Mean-Field and Online Inference**: This work introduces a novel approach by integrating the Ensemble Kalman Filter with Gaussian process state-space models. The combination aims to resolve issues in variational inference, especially under non-mean-field assumptions, and supports online learning in real-time environments.

2. **The Geometric Unscented Kalman Filter**: This paper presents a nonlinear estimation methodology that uses a geometric unscented sampling strategy. It aims to offer stable and efficient performance akin to other Gaussian filtering methods while avoiding common issues like negative weights associated with the standard Unscented Kalman Filter.

3. **Accelerating the Spin-Up of Ensemble Kalman Filtering**: The paper introduces an enhancement for the initial performance of ensemble-based Kalman Filters, which allows for faster alignment with actual dynamics, thus reducing the time to reach optimal error levels.

4. **Feedback Capacity and a Variant of the Kalman Filter with ARMA Gaussian Noises**: This study examines a variant of the Kalman Filter in conjunction with autoregressive moving-average Gaussian noises, providing insights into feedback capacity and proposing recursive coding schemes to achieve these bounds effectively.

5. **Robust Extended Kalman Filter for Land Navigation Using Massive Array of MEMS IMUs**: This paper develops a Robust Inertial Sensor Array Fusion (RISAF) method. It efficiently aggregates data from MEMS sensors, addressing challenges like sensor bias errors and drift, enhancing navigation accuracy without increasing computational overhead.

### References Used

1. Zhidi Lin, Yiyong Sun, Feng Yin, Alexandre Hoang Thiéry, "Ensemble Kalman Filtering Meets Gaussian Process SSM for Non-Mean-Field and Online Inference," [arXiv:2312.05910v5](https://arxiv.org/pdf/2312.05910v5).

2. Chengling Fang, Jiang Liu, Songqing Ye, Ju Zhang, "The Geometric Unscented Kalman Filter," [arXiv:2009.13079v1](https://arxiv.org/pdf/2009.13079v1).

3. Eugenia Kalnay, Shu-Chih Yang, "Accelerating the spin-up of Ensemble Kalman Filtering," [arXiv:0806.0180v1](https://arxiv.org/pdf/0806.0180v1).

4. Song Fang, Quanyan Zhu, "Feedback Capacity and a Variant of the Kalman Filter with ARMA Gaussian Noises: Explicit Bounds and Feedback Coding Design," [arXiv:2001.03108v6](https://arxiv.org/pdf/2001.03108v6).

5. Omer Hanani, Alon Kipnis, "Robust Extended Kalman Filter for Land Navigation Using Massive Array of MEMS IMUs," [arXiv:2606.29271v1](https://arxiv.org/pdf/2606.29271v1).

## Without limiting steps

In [21]:
# If you want to see the full workflow without limiting the number of steps. Set limit_steps to False
# Keep in mind this could take more than 10 minutes to complete
executor_history = executor_agent("The ensemble Kalman filter for time series forecasting", limit_steps=False)

md = executor_history[-1][-1].strip("`")  
display(Markdown(md))

🎯 Editor Agent

🛠️ Executing with agent: `research_agent` on task: Search Wikipedia for the ensemble Kalman filter and retrieve a concise summary of its definition and key properties.
🔍 Research Agent
✅ Output:
 The Ensemble Kalman Filter (EnKF) is a recursive filter particularly suitable for applications involving a large number of variables, such as those found in geophysical models and problems involving partial differential equations. It is an extension of the Kalman filter designed for handling large-scale problems by replacing the covariance matrix with a sample covariance. EnKF is a key component in data assimilation processes, especially in ensemble forecasting.

Key Points:
- **Monte Carlo Implementation**: EnKF functions as a Monte Carlo method for Bayesian updates. It uses a Bayesian approach to update the probability density function (PDF) of the state after considering likelihoods from new data observations.
- **Approximation Method**: Unlike the standard Kalman filter, th


🛠️ Executing with agent: `research_agent` on task: Extract abstracts and key findings from the top 5 most cited papers identified.
🔍 Research Agent
✅ Output:
 Here are the abstracts and key findings from the top 5 most cited papers related to Ensemble Kalman Filter:

1. **Ensemble Kalman Filtering Meets Gaussian Process SSM for Non-Mean-Field and Online Inference**
   - **Authors**: Zhidi Lin, Yiyong Sun, Feng Yin, Alexandre Hoang Thiéry
   - **Published**: December 10, 2023
   - **Abstract**: This paper tackles challenges in Gaussian Process State-Space Models (GPSSM) by integrating the ensemble Kalman filter (EnKF) into non-mean-field (NMF) variational inference. By eliminating extensive parameterization, the approach results in a closed-form approximation of the evidence lower bound (ELBO) and facilitates better online learning applications. An enhanced inference performance is demonstrated across diverse datasets.
   - **Key Findings**: The integration improves inference accuracy,


🛠️ Executing with agent: `research_agent` on task: Summarize common algorithmic variations and extensions of the ensemble Kalman filter found in the literature.
🔍 Research Agent
✅ Output:
 ### Research Summary on Algorithmic Variations and Extensions of the Ensemble Kalman Filter

The Ensemble Kalman Filter (EnKF) has been extensively studied and modified to enhance its application in various domains. The following outlines notable algorithmic variations and extensions identified in the literature:

1. **Integration with Gaussian Process State-Space Models (GPSSM)**
   - **Study:** Zhidi Lin et al. integrated EnKF with GPSSM to address variational inference challenges under non-mean-field assumptions. This approach streamlines parameterization efforts, supports online learning, and achieves better data-fitting accuracy with model regularizations.
   - **Key Features:** Elimination of extensive parameterization, closed-form ELBO approximation, and improved inference performance through


🛠️ Executing with agent: `writer_agent` on task: Draft the Literature Review section, outlining key papers, algorithmic variations, and identified research gaps.
✍️ Writer Agent
✅ Output:
## Literature Review

The Ensemble Kalman Filter (EnKF) has been a focal point of substantial academic inquiry, offering significant innovations in data assimilation and forecasting methodologies. This literature review encapsulates major contributions to the development and application of EnKF, highlighting algorithmic variations and current gaps in research. 

### Key Contributions and Developments

1. **Integration with Gaussian Process State-Space Models (GPSSM)**:
   - **Study by Zhidi Lin et al. (2023):** This work seamlessly integrates EnKF with GPSSM to tackle challenges inherent in variational inference methods, particularly under non-mean-field assumptions. The integration enhances inference accuracy and supports efficient online learning. A key achievement of this study is the formulation 


🛠️ Executing with agent: `writer_agent` on task: Revise the Introduction and Literature Review based on the editor agent’s feedback.
✍️ Writer Agent
✅ Output:
## Revised Introduction

In recent years, the field of data assimilation has gained significant importance across various domains that require precise integration of observational data with computational models. Data assimilation refers to the iterative process of updating computational models with real-time observational data to enhance their predictive accuracy. This methodology is critical in scientific and engineering applications where forecasting precision and timely updates are essential.

### Background on Data Assimilation

Traditionally, data assimilation techniques have played pivotal roles in fields like climatology, meteorology, and oceanography, where models are used to simulate complex dynamical systems. These techniques aim to reduce discrepancies between model predictions and observed data, resulting in more ref


🛠️ Executing with agent: `writer_agent` on task: Draft the Methodology and Case Studies section, detailing algorithm implementation steps and application results.
✍️ Writer Agent
✅ Output:
## Methodology and Case Studies

### Methodology: Implementing the Ensemble Kalman Filter

The Ensemble Kalman Filter (EnKF) serves as a potent tool for assimilating observational data into computational models, particularly under conditions of dynamic uncertainty. Its strength lies in approximating the state distribution of a system using an ensemble of samples, which represents possible state vectors. This probabilistic approach efficiently handles high-dimensional, nonlinear systems.

#### Implementation Steps:

1. **Initialization**:
   - **Ensemble Generation**: Initialize an ensemble of state vectors. Each member of the ensemble represents a possible state of the system, incorporating initial uncertainty and variability.
   - **State Covariance Estimation**: Estimate the initial covariance mat

✅ Output:
## Conclusion and Future Work

### Conclusion

The exploration of the Ensemble Kalman Filter (EnKF) within the context of data assimilation and forecasting unveils its significant potential in enhancing the accuracy and efficiency of predictive models. Through a detailed examination of its methodology, as well as practical applications across diverse domains, this study confirms the transformative impact of EnKF on dynamic system forecasting.

**Key Findings**:
1. **Efficient Handling of High-Dimensional Systems**: EnKF's ability to approximate state distributions using an ensemble framework proves particularly advantageous in managing the complexities of high-dimensional, nonlinear systems. This is facilitated by its Monte Carlo sampling-based approach, which provides significant computational advantages over traditional filters.

2. **Robustness and Flexibility**: The flexibility of EnKF allows for effective integration with varied modeling frameworks, such as Gaussian Proc

Below is the complete research report formatted in Markdown, integrating all revised sections and references.

```markdown
# Exploring the Ensemble Kalman Filter: Advances and Applications in Data Assimilation and Forecasting

## Introduction

In recent years, the field of data assimilation has gained significant importance across various domains that require precise integration of observational data with computational models. Data assimilation refers to the iterative process of updating computational models with real-time observational data to enhance their predictive accuracy. This methodology is critical in scientific and engineering applications where forecasting precision and timely updates are essential.

### Background on Data Assimilation

Traditionally, data assimilation techniques have played pivotal roles in fields like climatology, meteorology, and oceanography, where models are used to simulate complex dynamical systems. These techniques aim to reduce discrepancies between model predictions and observed data, resulting in more refined and accurate models. The Kalman filter is one such traditional method, leveraging statistical techniques to update model states based on new data, minimizing prediction errors. However, the standard Kalman filter often struggles with scalability and computational feasibility when applied to high-dimensional systems prevalent in real-world scenarios.

### Motivation for Using the Ensemble Kalman Filter in Time Series Forecasting

The ensemble Kalman filter (EnKF) emerges as a significant advancement over its traditional counterpart, addressing challenges faced in systems with numerous variables and inherent non-linearities. EnKF reformulates the Kalman filter approach by using an ensemble of system state vectors, enabling approximation of state distributions even in large-scale, complex environments. This ensemble-based approach reduces computational demands while maintaining accuracy, making it particularly suitable for high-dimensional time series forecasting.

In time series forecasting, managing uncertainty and variability is crucial. The EnKF provides a robust framework for addressing these challenges by combining Monte Carlo sampling with Bayesian inference. It effectively estimates state variables under uncertainty, offering distinct advantages for dynamic system predictions. This capability facilitates reliable forecasts, accommodating the unpredictability often present in time series datasets.

Beyond computational efficiency, EnKF's adaptability and systematic update mechanism for continuous model refinement make it invaluable in real-time applications. Specifically, in rapidly evolving fields like weather prediction, the ability to quickly assimilate new data into predictive models can lead to more accurate forecasts and better decision-making processes.

In conclusion, the ensemble Kalman filter is a transformative tool in data assimilation, offering improvements in time series forecasting for dynamic, data-intensive environments. This introduction sets the stage for an in-depth exploration of EnKF methodologies, applications, and its impact on forecasting practices across various domains. The subsequent sections will delve into recent advancements, alternative algorithmic implementations, and practical applications of EnKF, illustrating its flexibility and power in contemporary scientific inquiry.

## Literature Review

The Ensemble Kalman Filter (EnKF) continues to be a focal point of extensive academic research, bringing significant innovations to data assimilation and forecasting methodologies. This literature review highlights major contributions, algorithmic variations, and identifies current research gaps in the field.

### Key Contributions and Developments

1. **Integration with Gaussian Process State-Space Models (GPSSM)**:
   - **Study by Zhidi Lin et al. (2023):** Integration of EnKF with GPSSM addresses variational inference challenges, particularly under non-mean-field assumptions. This approach enhances inference accuracy and supports efficient online learning, notably achieving a closed-form approximation of the evidence lower bound (ELBO), which increases EnKF's utility in high-dimensional systems.
   - [Link to Study](https://arxiv.org/abs/2312.05910v5)
   
2. **The Geometric Unscented Kalman Filter (GUF)**:
   - **Study by Chengling Fang et al. (2020):** GUF employs a geometric unscented sampling strategy, improving stability and accuracy in nonlinear state estimations. EnKF principles play a crucial role in maintaining efficient computations and enhancing precision.
   - [Link to Study](https://arxiv.org/abs/2009.13079v1)
   
3. **Spin-up Acceleration for EnKF**:
   - **Work by Eugenia Kalnay and Shu-Chih Yang (2008):** This study proposes a method to accelerate the initial spin-up phase of ensemble-based Kalman Filters using a no-cost ensemble Kalman Smoother, resulting in faster convergence and reduced initialization errors.
   - [Link to Study](https://arxiv.org/abs/0806.0180v1)
   
4. **Kalman Filtering in Adverse Sensor Conditions**:
   - **Research by Omer Hanani and Alon Kipnis (2026):** This paper explores robust EnKF architectures for improving sensor-based navigation systems, with frameworks like RISAF improving azimuth accuracy and reducing drift, exemplifying EnKF's adaptability to real-world challenges.
   - [Link to Study](http://arxiv.org/abs/2606.29271v1)

5. **Kalman Filter Variants for Communication Systems**:
   - **Study by Song Fang and Quanyan Zhu (2020):** Focuses on a Kalman filter variant for handling autocorrelated noise in communication feedback channels, highlighting EnKF's potential in enhancing feedback capacity in noisy environments.
   - [Link to Study](https://arxiv.org/abs/2001.03108v6)

### Algorithmic Variations and Extensions

Recent studies have introduced modifications to EnKF, enhancing its resilience and adaptability:

- **Hybrid Approaches:** Integrating EnKF with methods like the Gaussian Process leverages strengths from both frameworks, providing robust predictions under uncertainty.
- **Nonlinear Extensions:** Innovations such as the Geometric Unscented Kalman Filter address nonlinearities more effectively, emphasizing EnKF's flexibility.
- **Adaptive Data Structures:** Enhanced strategies for real-time data handling, particularly during model spin-up phases, have been proposed to improve computational efficiency and prediction accuracy.

### Identified Research Gaps

Despite these advancements, several gaps remain:

1. **High-Dimensional Nonlinear Systems**: Extending EnKF's effectiveness in highly nonlinear systems, particularly under limited computational resources, is critical.
2. **Scalability in Real-Time Applications**: Ensuring real-time applicability of EnKF as system dimensionality increases, without scalability issues, is imperative.
3. **Integration with Machine Learning Architectures**: While EnKF adaptations exist for some machine learning paradigms, deeper integration and exploration of joint frameworks with advanced AI technologies remain underexplored.
4. **Robustness to Model Errors**: Enhancements improving EnKF's robustness against model-induced errors or inaccuracies are necessary given the complexity of observational data.

This review reveals that while EnKF has significantly advanced data assimilation methodologies, ongoing research is crucial to address these gaps, particularly in enhancing its applicability and efficiency. The synergy between EnKF and emerging computational techniques presents a promising avenue for future exploration.

## Methodology and Case Studies

### Methodology: Implementing the Ensemble Kalman Filter

The Ensemble Kalman Filter (EnKF) serves as a potent tool for assimilating observational data into computational models, particularly under conditions of dynamic uncertainty. Its strength lies in approximating the state distribution of a system using an ensemble of samples, which represents possible state vectors. This probabilistic approach efficiently handles high-dimensional, nonlinear systems.

#### Implementation Steps:

1. **Initialization**:
   - **Ensemble Generation**: Initialize an ensemble of state vectors. Each member of the ensemble represents a possible state of the system, incorporating initial uncertainty and variability.
   - **State Covariance Estimation**: Estimate the initial covariance matrix using the ensemble to capture the initial uncertainty.
   
2. **Prediction Phase**:
   - **State Propagation**: Each ensemble member is propagated through the model dynamics. This step predicts the system’s state at the next time step using the governing equations of the model.
   - **Covariance Update**: The covariance matrix is updated to reflect the predicted ensemble spread, providing a probabilistic forecast of the system's future state.
   
3. **Analysis Phase**:
   - **Kalman Gain Calculation**: Compute the Kalman Gain matrix, which balances the weight between model predictions and new observations. It combines the predicted state information with the observed data to minimize error.
   - **Observation Assimilation**: Integrate observational data into the ensemble. Update each ensemble member using the Kalman Gain, adjusting them towards the observed data while considering measurement noise and model errors.
   - **Covariance Re-estimation**: Re-estimate the covariance matrix post-assimilation to represent the updated state uncertainty.

4. **Iterative Update**:
   - Repeat the prediction and analysis phases as new observational data becomes available, perpetually refining the model forecast.

### Case Studies: Application Results

The effectiveness of EnKF is demonstrated across various domains, notably where accurate, real-time forecasting is crucial. Below, we present case studies showcasing EnKF's capacities.

#### Case Study 1: Integration with Gaussian Process State-Space Models (GPSSM)

- **Application**: Zhidi Lin et al. (2023) explore integrating EnKF with GPSSM to enhance online learning in dynamic systems. This integration focuses on addressing variational inference challenges in non-mean-field scenarios.
- **Results**: The study demonstrates significant improvements in data-fitting accuracy and model regularization across diverse datasets. The EnKF effectively streamlines parameterization efforts and improves inference accuracy.

#### Case Study 2: Geometric Unscented Kalman Filter (GUF)

- **Application**: The Geometric Unscented Kalman Filter (Chengling Fang et al., 2020) incorporates elements of EnKF to address limitations in nonlinear state estimation.
- **Results**: The GUF maintains computational efficiency while enhancing prediction stability and accuracy, demonstrating superior performance compared to traditional UKF and CKF methodologies.

#### Case Study 3: Accelerating Spin-up for High-Frequency Forecast Models

- **Application**: Accelerating the initial spin-up of EnKF, as explored by Eugenia Kalnay and Shu-Chih Yang (2008), involves using the ensemble Kalman Smoother.
- **Results**: This method achieves rapid convergence to optimal error levels, crucial for high-frequency models, demonstrating reduced computational overhead and improved early prediction accuracy.

These case studies attest to the flexibility and robustness of EnKF in addressing complex forecasting scenarios. By efficiently managing uncertainty and incorporating real-time data, EnKF significantly enhances predictive accuracy and adaptability—critical in fields demanding timely and precise forecasts.

### Conclusion

Through a structured implementation and validation via practical case studies, the Ensemble Kalman Filter proves to be an indispensable technique in modern data assimilation contexts. Its versatility and computational efficiency offer substantial improvements over traditional methods, marking it as an invaluable asset for contemporary forecasting challenges. Future explorations might consider deeper integration with machine learning technologies and further refinements to enhance scalability and robustness in diverse applications.

## Conclusion and Future Work

### Conclusion

The exploration of the Ensemble Kalman Filter (EnKF) within the context of data assimilation and forecasting unveils its significant potential in enhancing the accuracy and efficiency of predictive models. Through a detailed examination of its methodology, as well as practical applications across diverse domains, this study confirms the transformative impact of EnKF on dynamic system forecasting.

**Key Findings**:
1. **Efficient Handling of High-Dimensional Systems**: EnKF's ability to approximate state distributions using an ensemble framework proves particularly advantageous in managing the complexities of high-dimensional, nonlinear systems. This is facilitated by its Monte Carlo sampling-based approach, which provides significant computational advantages over traditional filters.

2. **Robustness and Flexibility**: The flexibility of EnKF allows for effective integration with varied modeling frameworks, such as Gaussian Process State-Space Models, thereby enhancing inference accuracy for dynamic systems. Its iterative update mechanism ensures consistent model refinement, critical for real-time applications where rapid assimilation of new data is essential.

3. **Improved Forecast Accuracy**: Case studies have highlighted EnKF’s capacity to maintain accuracy across different applications, from weather prediction to nonlinear state estimation. Notably, the Geometric Unscented Kalman Filter and spin-up acceleration methods demonstrate EnKF's adaptability and efficacy in diverse forecasting challenges.

**Applications and Implications**:
The EnKF has proven its application beyond traditional domains, expanding into areas such as robust sensor navigation and communication systems. These applications underscore EnKF's broad potential in addressing forecast uncertainties across various fields, marking it as a pivotal tool in modern scientific inquiry.

### Future Work

While the current study substantiates EnKF's utility and effectiveness, several avenues for future research can further enhance its application and adaptability:

1. **Integration with Machine Learning**: There lies a promising opportunity in integrating EnKF with advanced machine learning architectures, such as deep neural networks. This integration could refine state estimation processes and leverage EnKF's uncertainty handling capabilities in data-intensive AI applications.

2. **Scalability Improvements**: Further investigation into optimizing EnKF for extremely large-scale systems is warranted. Enhancing scalability while minimizing computational load could expand its applicability to even more complex systems, such as global climate models.

3. **Enhanced Nonlinear System Handling**: Development of new methodologies or hybrid approaches that improve EnKF's performance in highly nonlinear environments should be pursued. Advances in non-linear approximation techniques could further bolster EnKF's accuracy and efficiency.

4. **Error Robustness**: Strengthening EnKF's robustness against model and observational errors remains critical. Future research should delve into error-correction mechanisms or adaptive filtering strategies that can effectively manage inconsistencies inherent in real-world data.

5. **Operational Deployment**: Scaling EnKF for operational use in industries such as aviation, maritime, and financial forecasting could provide valuable real-world validation. Case studies focusing on these applications could offer new insights into practical implementation challenges and benefits.

In conclusion, the Ensemble Kalman Filter stands as a cornerstone technique in the evolution of data assimilation strategies. Its continued development and application promise significant contributions to the realm of predictive modeling, ensuring precise and adaptive responses to an ever-changing environment. Through ongoing research and innovation, EnKF's potential can be further unlocked, addressing the dynamic needs of complex systems in the future.
```

This Markdown document incorporates all revised sections and references, providing a complete outline of the research on the Ensemble Kalman Filter.

## Check grading feedback

If you have collapsed the right panel to have more screen space for your code, as shown below:

<img src="./images/collapsed.png" alt="Collapsed Image" width="800" height="400"/>

You can click on the left-facing arrow button (highlighted in red) to view feedback for your submission after submitting it for grading. Once expanded, it should display like this:

<img src="./images/expanded.png" alt="Expanded Image" width="800" height="400"/>